# 🧠 MeetingMind AI — Pure Dual-GPU Kaggle Notebook
### GPU: NVIDIA Tesla T4 x2 (100% Dedicated GPU — 0% CPU)
### 🎤 Whisper: `large-v3` (GPU 0) | 🧠 NLP: `Qwen2.5-3B-Instruct` (GPU 1)

---
**Full Pipeline Architecture:**
1. 🎤 **Speech-to-Text** — Faster-Whisper `large-v3` on **T4 GPU 0** (FP16, beam_size=5)
2. 🧠 **NLP Engine** — `Qwen/Qwen2.5-3B-Instruct` on **T4 GPU 1** (FP16)
3. 📄 **Report Generation** — JSON + PDF Executive Meeting Report

> ⚙️ **Settings:** Kaggle → Settings → Accelerator → **GPU T4 x2** | Internet → **On**


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install faster-whisper transformers torch huggingface_hub av reportlab colorlog -q
!apt-get install -y ffmpeg -q
print("✅ All dependencies installed successfully!")


## 🔍 Step 2 — Verify Dedicated Dual-GPU (T4 x2)

In [ ]:
import torch

print("=" * 60)
print("  MeetingMind AI — Hardware Verification")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU found! Go to Kaggle Settings -> Accelerator -> GPU T4 x2.")

gpu_count = torch.cuda.device_count()
print(f"CUDA Available: YES ({gpu_count} GPUs detected)\n")
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} | VRAM: {vram:.1f} GB")

print("\nDual-GPU Allocation:")
print("  🎯 GPU 0 (cuda:0) -> Dedicated to Faster-Whisper large-v3")
print("  🎯 GPU 1 (cuda:1) -> Dedicated to Qwen2.5-3B-Instruct NLP Engine")
print("=" * 60)


## 📁 Step 3 — Clone / Update MeetingMind AI from GitHub

In [ ]:
import os, sys

GITHUB_REPO_URL = "https://github.com/nishupatel14/MeetingMind-AI.git"
PROJECT_DIR     = "/kaggle/working/MeetingMind-AI"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    os.system(f"git clone {GITHUB_REPO_URL} {PROJECT_DIR}")
else:
    print("Pulling latest updates from repository...")
    os.system(f"cd {PROJECT_DIR} && git pull origin main")

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"✅ MeetingMind AI ready: {PROJECT_DIR}")


## ⚙️ Step 4 — Pure GPU Configuration (3B Model on GPU 1)

In [ ]:
import os, sys, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

KAGGLE_CONFIG = {
    "WHISPER_MODEL"        : "large-v3",
    "WHISPER_DEVICE"       : "cuda",
    "WHISPER_DEVICE_INDEX" : 0,
    "WHISPER_COMPUTE_TYPE" : "float16",
    "WHISPER_BEAM_SIZE"    : 5,
    "WHISPER_LANGUAGE"     : "en",
    "WHISPER_VAD_FILTER"   : True,
    "WHISPER_MIN_SILENCE_MS": 500,
    "ACTION_MODEL"         : "Qwen/Qwen2.5-3B-Instruct",
    "DEVICE"               : "cuda",
    "HF_DEVICE"            : 1,
    "TORCH_DTYPE"          : torch.float16,
}

print("=" * 60)
print("  MeetingMind AI — Dual-GPU Configuration")
print("=" * 60)
print("  Whisper Model  : " + KAGGLE_CONFIG["WHISPER_MODEL"] + " on GPU 0 (cuda:0)")
print("  NLP Model      : " + KAGGLE_CONFIG["ACTION_MODEL"] + " on GPU 1 (cuda:1)")
print("=" * 60)


## 🔧 Step 5 — Apply GPU Settings to AI Engine

In [ ]:
import ai_engine.config as cfg

cfg.WHISPER_MODEL                 = KAGGLE_CONFIG["WHISPER_MODEL"]
cfg.WHISPER_DEVICE                = KAGGLE_CONFIG["WHISPER_DEVICE"]
cfg.WHISPER_DEVICE_INDEX          = KAGGLE_CONFIG["WHISPER_DEVICE_INDEX"]
cfg.WHISPER_COMPUTE_TYPE          = KAGGLE_CONFIG["WHISPER_COMPUTE_TYPE"]
cfg.WHISPER_BEAM_SIZE             = KAGGLE_CONFIG["WHISPER_BEAM_SIZE"]
cfg.WHISPER_LANGUAGE              = KAGGLE_CONFIG["WHISPER_LANGUAGE"]
cfg.WHISPER_VAD_FILTER            = KAGGLE_CONFIG["WHISPER_VAD_FILTER"]
cfg.WHISPER_MIN_SILENCE_DURATION_MS = KAGGLE_CONFIG["WHISPER_MIN_SILENCE_MS"]

cfg.ACTION_MODEL   = KAGGLE_CONFIG["ACTION_MODEL"]
cfg.DEVICE         = KAGGLE_CONFIG["DEVICE"]
cfg.HF_DEVICE      = KAGGLE_CONFIG["HF_DEVICE"]
cfg.TORCH_DTYPE    = KAGGLE_CONFIG["TORCH_DTYPE"]

cfg.EXEC_MODE_STR = (
    "Dual-GPU [Whisper=" + cfg.WHISPER_MODEL + " on GPU:0] [NLP=" + cfg.ACTION_MODEL + " on GPU:1]"
)
print("✅ Config applied: " + cfg.EXEC_MODE_STR)


## 🎤 Step 6 — Load Faster-Whisper large-v3 on GPU 0

In [ ]:
import gc, torch
from ai_engine.speech.whisper_model import WhisperLoader

gc.collect()
torch.cuda.empty_cache()

print("Loading Faster-Whisper large-v3 on GPU 0 (cuda:0)...")
whisper_model = WhisperLoader.get_model()

used0 = torch.cuda.memory_allocated(0) / 1024**3
tot0  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nGPU 0 VRAM: {used0:.2f} GB / {tot0:.1f} GB (Whisper large-v3 loaded)")


## 🧠 Step 7 — Load Qwen2.5-3B-Instruct on GPU 1

In [ ]:
import gc, torch
from ai_engine.nlp.action_model import ActionModelLoader

# 1. Wipe GPU 1 memory to ensure zero old models remain
ActionModelLoader.unload()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# 2. Load 3B model directly into GPU 1
ActionModelLoader.load()

# 3. Verify VRAM status
used1 = torch.cuda.memory_allocated(1) / 1024**3
tot1  = torch.cuda.get_device_properties(1).total_memory / 1024**3
free1 = tot1 - used1

print(f"\nGPU 1 VRAM Used : {used1:.2f} GB / {tot1:.1f} GB")
print(f"GPU 1 VRAM Free : {free1:.2f} GB (Headroom for generation)")
print("✅ Qwen2.5-3B-Instruct ready on GPU 1!")


## 📂 Step 8 — Set Audio File Path

In [ ]:
import os

AUDIO_FILE  = "/kaggle/input/datasets/serverip/meeting-audio/Game Zone 2.m4a"
OUTPUT_NAME = "game_zone_meeting_001"

if not os.path.exists(AUDIO_FILE):
    found = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith(('.mp3', '.wav', '.m4a', '.flac')):
                found.append(os.path.join(root, f))
    if found:
        AUDIO_FILE = found[0]
        print(f"Found audio: {AUDIO_FILE}")
    else:
        raise FileNotFoundError("Audio file not found in /kaggle/input.")

size_mb = os.path.getsize(AUDIO_FILE) / (1024 * 1024)
print(f"Audio File  : {AUDIO_FILE} ({size_mb:.1f} MB)")
print(f"Output Name : {OUTPUT_NAME}")


## 🔄 Step 9 — Convert Audio to 16 kHz Mono WAV

In [ ]:
import subprocess

WAV_OUTPUT = f"/kaggle/working/{OUTPUT_NAME}.wav"
cmd = [
    "ffmpeg", "-y", "-i", AUDIO_FILE,
    "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le",
    WAV_OUTPUT,
]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode == 0 and os.path.exists(WAV_OUTPUT):
    wav_mb = os.path.getsize(WAV_OUTPUT) / (1024 * 1024)
    print(f"✅ WAV Created: {WAV_OUTPUT} ({wav_mb:.1f} MB)")
else:
    raise RuntimeError(f"Audio conversion failed: {res.stderr[-400:]}")


## 🎙️ Step 10 — Transcribe Audio on GPU 0

In [ ]:
import time, json

print("=" * 60)
print("  Transcribing on GPU 0 (Faster-Whisper large-v3)")
print("=" * 60)

start_t = time.time()
transcript_lines = []
transcript_json  = []

segments, info = whisper_model.transcribe(
    WAV_OUTPUT,
    language=KAGGLE_CONFIG["WHISPER_LANGUAGE"],
    beam_size=KAGGLE_CONFIG["WHISPER_BEAM_SIZE"],
    vad_filter=KAGGLE_CONFIG["WHISPER_VAD_FILTER"],
    vad_parameters={"min_silence_duration_ms": KAGGLE_CONFIG["WHISPER_MIN_SILENCE_MS"]},
)

print(f"Language: {info.language} | Duration: {info.duration:.1f}s ({info.duration/60:.1f} min)")
print("-" * 60)

for seg in segments:
    text = str(seg.text).strip()
    if text:
        print(f"[{seg.start:.2f} - {seg.end:.2f}] {text}")
        transcript_lines.append(text)
        transcript_json.append({"start": round(seg.start, 2), "end": round(seg.end, 2), "text": text})

elapsed = time.time() - start_t
transcript_str = "\n".join(transcript_lines)

os.makedirs("/kaggle/working/output", exist_ok=True)
with open(f"/kaggle/working/output/{OUTPUT_NAME}_transcript.txt", "w", encoding="utf-8") as f:
    f.write(transcript_str)
with open(f"/kaggle/working/output/{OUTPUT_NAME}_transcript.json", "w", encoding="utf-8") as f:
    json.dump(transcript_json, f, indent=4, ensure_ascii=False)

print("-" * 60)
print(f"✅ Transcribed {len(transcript_lines)} segments in {elapsed:.1f}s")


## 🧠 Step 11 — Full NLP Analysis on GPU 1 (All 7 Tasks)

In [ ]:
import gc, json, torch

def safe_empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_header(title):
    line = "=" * 60
    print("\n" + line + "\n  " + title + "\n" + line)

results = {}

# 1. Executive Summary
print_header("1/7 — Executive Summary (GPU 1)")
from ai_engine.nlp.summary_generator import SummaryGenerator
results["summary"] = SummaryGenerator().summarize(transcript_str)
print(results["summary"])
safe_empty_cache()

# 2. Discussion Topics
print_header("2/7 — Discussion Topics (GPU 1)")
from ai_engine.nlp.topic_detector import TopicDetector
results["topics"] = TopicDetector().detect_topics(transcript_str)
print(results["topics"])
safe_empty_cache()

# 3. Key Decisions
print_header("3/7 — Key Decisions (GPU 1)")
from ai_engine.nlp.decision_detector import DecisionDetector
results["decisions"] = DecisionDetector().detect(transcript_str)
print(results["decisions"])
safe_empty_cache()

# 4. Action Items
print_header("4/7 — Action Items (GPU 1)")
from ai_engine.nlp.action_items import ActionItemExtractor
results["actions"] = ActionItemExtractor().extract(transcript_str)
print(results["actions"])
safe_empty_cache()

# 5. Open Questions
print_header("5/7 — Open Questions (GPU 1)")
from ai_engine.nlp.open_questions_detector import OpenQuestionsDetector
results["open_questions"] = OpenQuestionsDetector().detect(transcript_str)
print(results["open_questions"])
safe_empty_cache()

# 6. Key Insights
print_header("6/7 — Key Insights (GPU 1)")
from ai_engine.nlp.key_insights_detector import KeyInsightsDetector
results["key_insights"] = KeyInsightsDetector().detect(transcript_str)
print(results["key_insights"])
safe_empty_cache()

# 7. Key Discussion Points (Timestamped)
print_header("7/7 — Key Discussion Points (Timestamped, GPU 1)")
from ai_engine.nlp.key_discussion_detector import KeyDiscussionDetector
results["key_discussion"] = KeyDiscussionDetector().detect(transcript_json)
print(json.dumps(results["key_discussion"], indent=2, ensure_ascii=False))
safe_empty_cache()

print("\n" + "=" * 60)
print("  ✅ All 7 NLP Analysis Tasks Completed Successfully on GPU 1!")
print("=" * 60)


## 💾 Step 12 — Generate JSON & PDF Reports

In [ ]:
from ai_engine.config import (
    SUMMARY_FOLDER, TOPICS_FOLDER, DECISIONS_FOLDER,
    ACTION_ITEMS_FOLDER, OPEN_QUESTIONS_FOLDER,
    KEY_INSIGHTS_FOLDER, KEY_DISCUSSION_FOLDER,
    TRANSCRIPT_FOLDER,
)

def save_to_folder(folder, filename, content):
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    with open(path, 'w', encoding='utf-8') as f:
        if isinstance(content, str):
            f.write(content)
        else:
            json.dump(content, f, indent=4, ensure_ascii=False)

save_to_folder(TRANSCRIPT_FOLDER,     f"{OUTPUT_NAME}.txt",                 "\n".join(transcript_lines))
save_to_folder(TRANSCRIPT_FOLDER,     f"{OUTPUT_NAME}.json",                transcript_json)
save_to_folder(SUMMARY_FOLDER,        f"{OUTPUT_NAME}_summary.txt",         results.get("summary",""))
save_to_folder(TOPICS_FOLDER,         f"{OUTPUT_NAME}_topics.txt",          results.get("topics",""))
save_to_folder(DECISIONS_FOLDER,      f"{OUTPUT_NAME}_decisions.txt",       results.get("decisions",""))
save_to_folder(ACTION_ITEMS_FOLDER,   f"{OUTPUT_NAME}_action_items.txt",    results.get("actions",""))
save_to_folder(OPEN_QUESTIONS_FOLDER, f"{OUTPUT_NAME}_open_questions.txt",  results.get("open_questions",""))
save_to_folder(KEY_INSIGHTS_FOLDER,   f"{OUTPUT_NAME}_key_insights.txt",    results.get("key_insights",""))
save_to_folder(KEY_DISCUSSION_FOLDER, f"{OUTPUT_NAME}_key_discussion.json", results.get("key_discussion",[]))

print("Generating JSON meeting report...")
from ai_engine.report.report_generator import ReportGenerator
ReportGenerator().generate(OUTPUT_NAME)
print("✅ JSON report generated.")

print("\nGenerating PDF meeting report...")
try:
    from ai_engine.report.pdf_report_generator import PDFReportGenerator
    PDFReportGenerator().generate(OUTPUT_NAME)
    print("✅ PDF report generated.")
except Exception as e:
    print(f"PDF note: {e}")

print("\n🎉 MeetingMind AI Pure GPU Pipeline Complete!")


## 📊 Step 13 — GPU VRAM Usage & Output Files

In [ ]:
import torch, os

print("=" * 60)
print("  Dual-GPU Dedicated Memory Status")
print("=" * 60)
roles = ["GPU 0: Faster-Whisper large-v3", "GPU 1: Qwen2.5-3B-Instruct"]
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    role  = roles[i] if i < len(roles) else f'GPU {i}'
    bar   = int((used / total) * 30)
    bar_str = '█' * bar + '░' * (30 - bar)
    print(f"  {role:<35} : {used:.2f}/{total:.1f} GB [{bar_str}] {used/total*100:.0f}% VRAM")

print("\n" + "=" * 60)
print("  Output Files in /kaggle/working/output/")
print("=" * 60)
OUTPUT_DIR = "/kaggle/working/output"
if os.path.isdir(OUTPUT_DIR):
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        kb    = os.path.getsize(fpath) / 1024
        icon  = '📄' if fname.endswith('.txt') else '📦' if fname.endswith('.json') else '📋'
        print(f"  {icon}  {fname:<50} {kb:>7.1f} KB")

print("\n💡 Download outputs from Kaggle sidebar -> /kaggle/working/output/")
